In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
from pt_to_api import disjoint_ae
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment

In [ ]:
import warnings

MODE = "light"


def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]


def generate_synthetic_patches(
    patch_dim=72, n_components=10, k=3, n_samples=1000, noise_std=0.01, seed=42
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses exactly k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        idx = rng.choice(n_components, k, replace=False)
        codes_true[i, idx] = rng.randn(k)

    X = codes_true @ W_true
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition


def show_closest_component_of_W_for_each_component(components, W_true, figsize=(5, 2)):
    """
    Given two arrays of numpy vectors of same shapes
    for every component in `components`, this function shows the array in `W_true`
    which has the maximum cosine similarity with the component
    """
    sims = np.abs(cosine_similarity(components, W_true))
    pairs = []
    for i in range(len(components)):
        j = np.argmax(sims[i])
        pairs.append((i, j, sims[i][j]))
    for i, j, score in pairs:
        S(
            [components[i].reshape(3, 3), W_true[j].reshape(3, 3)],
            figsize,
            mode=MODE,
            suptitle=f"similarity score={score}",
            ax_titles=["component", "ground_truth"],
            viztype="local",
        )
        plt.show()


def evaluate_recovery(W_learned, W_true, threshold=0.95):
    """
    W_learned: (n_atoms, patch_dim)
    W_true: (n_atoms, patch_dim)
    
    for each true atom, finds the best matching learned atom by cosine similarity
    returns fraction of true atoms recovered above threshold
    """
    W_l = W_learned / (np.linalg.norm(W_learned, axis=1, keepdims=True) + 1e-8)
    W_t = W_true / (np.linalg.norm(W_true, axis=1, keepdims=True) + 1e-8)
    
    sim = np.abs(W_l @ W_t.T)  # (n_atoms, n_atoms), abs because sign is arbitrary
    best_match = sim.max(axis=0)  # for each true atom, best cosine with any learned atom
    
    recovered = (best_match >= threshold).mean()
    print(f"Mean best cosine similarity: {best_match.mean():.4f}")
    print(f"Fraction recovered (>{threshold}): {recovered:.4f}")
    return best_match, recovered


def _match_atoms(D1: np.ndarray, D2: np.ndarray):
    """
    Match atoms of D1 to atoms of D2 using the Hungarian algorithm
    on cosine distances. Assumes square dictionaries (same n_components).

    D1, D2: shape (n_components, n_features) — sklearn's components_ layout.

    Returns:
        row_ind, col_ind: matched index arrays
        matched_similarities: per-pair cosine similarities
        mean_sim: mean cosine similarity across matched pairs
    """
    # Guard against dead atoms (zero-norm rows produce NaN cosine distances)
    norms_1 = np.linalg.norm(D1, axis=1, keepdims=True)
    norms_2 = np.linalg.norm(D2, axis=1, keepdims=True)
    if np.any(norms_1 == 0) or np.any(norms_2 == 0):
        raise ValueError(
            "One or more atoms have zero norm. "
            "Remove or replace dead atoms before matching."
        )

    # cost = cosine_distances(D1, D2)          # shape (n_components, n_components), values in [0, 2]
    cost = np.abs(cosine_similarity(D1, D2))
    cost = 1 - cost

    row_ind, col_ind = linear_sum_assignment(cost)
    matched_similarities = 1 - cost[row_ind, col_ind]
    mean_sim = float(matched_similarities.mean())
    return row_ind, col_ind, matched_similarities, mean_sim

def get_live(components):
    dead = find_dead_atoms(components).numpy()
    live = np.array([i for i in range(components.shape[0]) if i not in dead])
    return components[live]

def hungarian_match(all_components: list[np.ndarray]):
    """
    Pairwise similarity matching across runs using the Hungarian algorithm.

    Args:
        all_components: list of arrays, each shape (n_components, n_features).
                        All arrays must have the same shape.

    Returns:
        upper: 1-D array of pairwise similarities for all unique pairs
        stability_score: mean of upper
        best_run_idx: index of the run most similar to all others
        pairwise_sims: (n_runs, n_runs) symmetric similarity matrix, diagonal = 1
    """
    n_runs = len(all_components)

    if n_runs < 2:
        raise ValueError("Need at least 2 runs to compute pairwise similarity.")

    shapes = [d.shape for d in all_components]
    if len(set(shapes)) != 1:
        raise ValueError(
            f"All dictionaries must have the same shape. Got: {shapes}"
        )

    pairwise_sims = np.ones((n_runs, n_runs))
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            _, _, _, mean_sim = _match_atoms(all_components[i], all_components[j])
            pairwise_sims[i, j] = mean_sim
            pairwise_sims[j, i] = mean_sim

    upper = pairwise_sims[np.triu_indices(n_runs, k=1)]
    stability_score = float(upper.mean())

    # Exclude self-similarity (diagonal=1) when ranking runs
    np.fill_diagonal(pairwise_sims, 0)
    mean_sim_per_run = pairwise_sims.sum(axis=1) / (n_runs - 1)
    best_run_idx = int(np.argmax(mean_sim_per_run))
    np.fill_diagonal(pairwise_sims, 1)  # restore diagonal

    return upper, stability_score, best_run_idx, pairwise_sims

def find_dead_atoms(W, threshold=0.1):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)
    
    peak = W.abs().max(dim=1).values
    peak_normalised = peak / peak.max()
    
    return torch.where(peak_normalised < threshold)[0]

In [ ]:
from dataclasses import dataclass

@dataclass
class SingleRun:
    codes: np.ndarray
    components: np.ndarray
    recon: np.ndarray
    X: np.ndarray
    baseline_loss: float
    loss: float
    gram_error: float
    dead_atom_count: int

def plot_runs(runs: list[SingleRun]):
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    
    losses = [r.loss for r in runs]
    baselines = [r.baseline_loss for r in runs]
    gram_errors = [r.gram_error for r in runs]
    dead_counts = [r.dead_atom_count for r in runs]
    
    axes[0].plot(losses, color="red", label="loss")
    axes[0].plot(baselines, color="blue", label="baseline loss")
    axes[0].legend()
    axes[0].set_title("Loss")
    
    axes[1].plot(gram_errors, color="green")
    axes[1].set_title("Gram Error")
    
    axes[2].plot(dead_counts, color="orange")
    axes[2].set_title("Dead Atoms")
    
    axes[2].set_xlabel("Run")
    plt.tight_layout()
    plt.show()


def mse(x, recon):
    return ((x-recon)**2).sum()

def train_for(X, n_components, epochs=2000, baseline_epochs=600, verbose=False, svd_init=False):
    model, codes, components, recon = disjoint_ae.train_baseline(X, n_components, epochs=baseline_epochs, verbose=verbose)

    eps_tol = 1e-8
    std_eps = max((X - recon).std(), eps_tol)
    print("using eps", std_eps, "actual val", (X-recon).std())
    sigma_0 = std_eps * 10
    sigma_s = sigma_0 * 5
    alpha = 5000 / (sigma_0*sigma_0)

    baseline_loss = mse(X, recon)


    model, codes, components, recon = disjoint_ae.train(
        X, n_components, lr=1e-3, epochs=epochs, sigma_0=sigma_0, sigma_s=sigma_s, alpha=alpha, sigma_eps=std_eps, verbose=verbose, svd_init=svd_init
    )
    gram_error = gram_orthogonality_error(components.T)
    dead_atom_count = len(find_dead_atoms(components))

    loss = mse(X, recon)
    return SingleRun(
        codes, components, recon, X, baseline_loss, loss, gram_error, dead_atom_count
    )

In [ ]:
class AtomClusteringTieError(Exception):
    """
    Raised when a new atom matches multiple existing clusters at the same
    mean similarity score, making assignment ambiguous.

    Attributes:
        atom: the atom vector that caused the tie (1-D array)
        tied_cluster_indices: list of cluster indices that tied
        tied_scores: the tied mean similarity scores
        clusters: the full list of clusters at the time of the error
    """
    def __init__(self, atom, tied_cluster_indices, tied_scores, clusters):
        self.atom = atom
        self.tied_cluster_indices = tied_cluster_indices
        self.tied_scores = tied_scores
        self.clusters = clusters
        super().__init__(
            f"Atom matched {len(tied_cluster_indices)} clusters with identical "
            f"mean similarity {tied_scores[0]:.6f} "
            f"(cluster indices: {tied_cluster_indices}). "
            f"Inspect .atom, .tied_cluster_indices, .tied_scores, and .clusters "
            f"for details."
        )


def _atom_is_degenerate(atom: np.ndarray) -> bool:
    return float(np.linalg.norm(atom)) == 0.0


def _mean_sim_to_cluster(atom: np.ndarray, cluster: list[np.ndarray]) -> float:
    """Mean cosine similarity (abs) between atom and every member of cluster."""
    cluster_matrix = np.stack(cluster, axis=0)   # (k, n_features)
    sims = np.abs(cosine_similarity(atom.reshape(1, -1), cluster_matrix))  # (1, k)
    return float(sims.mean())


def find_stable_atoms(
    all_components: list[np.ndarray],
    svd_components: list[np.ndarray],
    threshold: float = 0.9,
) -> list[list[np.ndarray]]:
    """
    Cluster atoms across runs into groups of mutually similar atoms.

    Seeds clusters from best_run_idx, then processes remaining runs in order.
    Each cluster is a list of atom vectors that are all mutually similar
    (similarity > threshold).

    To find the most stable atoms, sort the returned list by cluster size
    (descending) — larger clusters = more stable across runs.

    Args:
        all_components: list of arrays, each shape (n_components, n_features).
        best_run_idx:   index into all_components to use as the seed run.
        threshold:      minimum mean cosine similarity to assign an atom to
                        an existing cluster.

    Returns:
        clusters: list of lists of np.ndarray (atom vectors).

    Raises:
        AtomClusteringTieError: if an atom ties across two or more clusters.
    """
    clusters: list[list[np.ndarray]] = []

    best_run_idx = None
    if svd_components is None:
        best_run_idx = random.randint(0, len(all_components)-1)
        first_components = all_components[best_run_idx]
    else:
        first_components = svd_components


    # --- seed from best run ---
    for atom in first_components:
        if _atom_is_degenerate(atom):
            warnings.warn(
                "Degenerate atom (zero norm) found in best run — skipping.",
                RuntimeWarning,
                stacklevel=2,
            )
            continue
        clusters.append([atom])

    # --- process remaining runs ---
    run_order = [i for i in range(len(all_components)) if i != best_run_idx]

    for run_idx in run_order:
        for atom in all_components[run_idx]:
            if _atom_is_degenerate(atom):
                warnings.warn(
                    f"Degenerate atom (zero norm) found in run {run_idx} — skipping.",
                    RuntimeWarning,
                    stacklevel=2,
                )
                continue

            # score against every existing cluster
            scores = [_mean_sim_to_cluster(atom, c) for c in clusters]

            best_score = max(scores)

            if best_score <= threshold:
                # no existing cluster is similar enough — start a new one
                clusters.append([atom])
                continue

            # check for ties among clusters that exceed threshold
            tied_indices = [
                i for i, s in enumerate(scores)
                if s > threshold and np.isclose(s, best_score)
            ]

            if len(tied_indices) > 1:
                raise AtomClusteringTieError(
                    atom=atom,
                    tied_cluster_indices=tied_indices,
                    tied_scores=[scores[i] for i in tied_indices],
                    clusters=clusters,
                )

            clusters[tied_indices[0]].append(atom)

    return clusters

def plot_sweep_summary(
    sweep: dict[int, list[SingleRun]],
    svd_runs: dict[int, SingleRun] = None,
):
    n_components_values = sorted(sweep.keys())

    losses, losses_std = [], []
    baseline_losses, baseline_losses_std = [], []
    gram_errors, gram_errors_std = [], []
    dead_counts, dead_counts_std = [], []

    for n in n_components_values:
        runs = sweep[n]
        losses.append(np.mean([r.loss for r in runs]))
        losses_std.append(np.std([r.loss for r in runs]))
        baseline_losses.append(np.mean([r.baseline_loss for r in runs]))
        baseline_losses_std.append(np.std([r.baseline_loss for r in runs]))
        gram_errors.append(np.mean([r.gram_error for r in runs]))
        gram_errors_std.append(np.std([r.gram_error for r in runs]))
        dead_counts.append(np.mean([r.dead_atom_count for r in runs]))
        dead_counts_std.append(np.std([r.dead_atom_count for r in runs]))

    losses, losses_std = np.array(losses), np.array(losses_std)
    baseline_losses, baseline_losses_std = np.array(baseline_losses), np.array(baseline_losses_std)
    gram_errors, gram_errors_std = np.array(gram_errors), np.array(gram_errors_std)
    dead_counts, dead_counts_std = np.array(dead_counts), np.array(dead_counts_std)

    svd_losses, svd_baseline_losses, svd_gram_errors, svd_dead_counts = None, None, None, None
    if svd_runs is not None:
        svd_losses = [svd_runs[n].loss for n in n_components_values]
        svd_baseline_losses = [svd_runs[n].baseline_loss for n in n_components_values]
        svd_gram_errors = [svd_runs[n].gram_error for n in n_components_values]
        svd_dead_counts = [svd_runs[n].dead_atom_count for n in n_components_values]

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    x = n_components_values

    # Loss + baseline loss
    axes[0].plot(x, losses, color="red", label="loss")
    axes[0].fill_between(x, losses - losses_std, losses + losses_std, alpha=0.2, color="red")
    axes[0].plot(x, baseline_losses, color="blue", label="baseline loss")
    axes[0].fill_between(x, baseline_losses - baseline_losses_std, baseline_losses + baseline_losses_std, alpha=0.2, color="blue")
    if svd_runs is not None:
        axes[0].plot(x, svd_losses, color="darkred", linestyle="--", label="svd loss")
        axes[0].plot(x, svd_baseline_losses, color="darkblue", linestyle="--", label="svd baseline")
    axes[0].set_title("Loss")
    axes[0].legend()

    # Gram error
    axes[1].plot(x, gram_errors, color="green", label="sweep mean")
    axes[1].fill_between(x, gram_errors - gram_errors_std, gram_errors + gram_errors_std, alpha=0.2, color="green")
    if svd_runs is not None:
        axes[1].plot(x, svd_gram_errors, color="black", linestyle="--", label="svd")
    axes[1].set_title("Gram Error")
    axes[1].legend()

    # Dead atoms
    axes[2].plot(x, dead_counts, color="orange", label="sweep mean")
    axes[2].fill_between(x, dead_counts - dead_counts_std, dead_counts + dead_counts_std, alpha=0.2, color="orange")
    if svd_runs is not None:
        axes[2].plot(x, svd_dead_counts, color="black", linestyle="--", label="svd")
    axes[2].set_title("Dead Atoms")
    axes[2].legend()

    axes[2].set_xlabel("n_components")
    plt.tight_layout()
    plt.show()


In [ ]:
# single atom perturbations

def gaussian_noise_perturbation(noise_std, ratio=1.0):
    def perturb(X, W_true, codes_true, dim_partition):
        rng = np.random.RandomState()
        n_samples = X.shape[0]
        n_affected = int(n_samples * ratio)
        idx = rng.choice(n_samples, n_affected, replace=False)
        X_out = X.copy()
        X_out[idx] += rng.randn(n_affected, X.shape[1]) * noise_std
        return X_out, W_true, codes_true, dim_partition
    return perturb

def replacement_perturbation(atom_index, ratio, per_sample=False):
    def perturb(X, W_true, codes_true, dim_partition):
        rng = np.random.RandomState()
        n_samples, patch_dim = X.shape
        n_affected = int(n_samples * ratio)
        idx = rng.choice(n_samples, n_affected, replace=False)

        dims = dim_partition[atom_index]

        def make_corrupt_atom():
            a = np.zeros(patch_dim)
            a[dims] = rng.randn(len(dims))
            a /= np.linalg.norm(a)
            return a

        if not per_sample:
            corrupt_atom = make_corrupt_atom()

        X_out = X.copy()
        for i in idx:
            coeff = codes_true[i, atom_index]
            if coeff == 0:
                continue
            ca = make_corrupt_atom() if per_sample else corrupt_atom
            X_out[i] -= coeff * W_true[atom_index]
            X_out[i] += coeff * ca

        return X_out, W_true, codes_true, dim_partition
    return perturb

def zeroing_perturbation(atom_index, ratio, n_dims_to_zero=None):
    def perturb(X, W_true, codes_true, dim_partition):
        rng = np.random.RandomState()
        n_samples, patch_dim = X.shape
        n_affected = int(n_samples * ratio)
        idx = rng.choice(n_samples, n_affected, replace=False)

        dims = dim_partition[atom_index]

        X_out = X.copy()
        for i in idx:
            coeff = codes_true[i, atom_index]
            if coeff == 0:
                continue

            k = n_dims_to_zero if n_dims_to_zero is not None else rng.randint(1, len(dims) + 1)
            dims_to_zero = rng.choice(dims, k, replace=False)

            # build zeroed atom
            zeroed_atom = W_true[atom_index].copy()
            zeroed_atom[dims_to_zero] = 0.0

            X_out[i] -= coeff * W_true[atom_index]
            X_out[i] += coeff * zeroed_atom

        return X_out, W_true, codes_true, dim_partition
    return perturb

In [ ]:
N_COMPONENTS = 6

In [ ]:
X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    patch_dim=9, n_components=N_COMPONENTS, k=3
)
ws_to_show = [w.reshape(3, 3) for w in W_true]
S(
    ws_to_show,
    (8, 2),
    ncols=N_COMPONENTS,
    mode=MODE,
    suptitle="ground truth basis vectors \n[bright green = high positive, bright red = high negative, white = near zero]\nA vector is of size 1x9, but is shown as 3x3 for easier visibility",
)
plt.show()
S(
    [X[0].reshape(3, 3)] + ws_to_show,
    (8, 2),
    N_COMPONENTS+1,
    suptitle="First input, with its basis components, the coefficient of each component is it's title",
    mode=MODE,
    ax_titles=["X"] + [f"{codes_true[1][k]:.4f}" for k in range(N_COMPONENTS)],
)
plt.show()
S(
    [X[1].reshape(3, 3)] + ws_to_show,
    (8, 2),
    N_COMPONENTS+1,
    suptitle="Second input",
    mode=MODE,
    ax_titles=["X"] + [f"{codes_true[1][k]:.4f}" for k in range(N_COMPONENTS)],
)
plt.show()

In [ ]:
runs = []
for comp in range(1, min(N_COMPONENTS+6, 9)):
    print(f"num comps: {comp}")
    runs.append(train_for(X, comp, epochs=4000))

In [ ]:
# hasnt converged
show_closest_component_of_W_for_each_component(runs[5].components, W_true)

In [ ]:
run = train_for(X, 6, 5000, 1000, True)
print(run)

# Plot for ds across seeds

In [ ]:


# 0th run is empty
all_runs = [[]]
all_runs = {}

for comp in range(1,9):
    print(f"#### comp {comp} ####")
    seed_runs = []
    for seed in range(10):
        np.random.seed(seed)
        torch.manual_seed(seed)
        seed_runs.append(
            train_for(X, comp)
        )
    all_runs[comp] = seed_runs

In [ ]:
svd_runs = {}
for comp in range(1,9):
    print(f"#### comp {comp} ####")
    svd_runs[comp] = train_for(X, comp, svd_init=True)

In [ ]:
hung_res = {}
for k in all_runs:
    _, stability_score, _, pairwise_sim = hungarian_match([r.components for r in all_runs[k]])
    hung_res[k] = (stability_score, pairwise_sim)

In [ ]:
# clusters = find_stable_atoms([r.components for r in all_runs.values()], None, 0.9)
# we have 4 runs of usefulness, 5-6-7-8
# lets get across them

import itertools

# all_comps = list(itertools.chain.from_iterable([[r.components for r in all_runs[k]] for k in [5,6,7]]))
clusters = {}
clusters[6] = find_stable_atoms([r.components for r in all_runs[6]], svd_runs[6].components)

In [ ]:
S([c.reshape(3,3) for c in W_true], (8,2), len(W_true), mode=MODE)
plt.show()

In [ ]:
S([c.reshape(3,3) for c in svd_runs[6].components], (8,2), len(W_true), mode=MODE)
plt.show()

In [ ]:
def cluster_internal_similarity(cluster: list[np.ndarray]) -> float:
    if len(cluster) < 2:
        return 1.0
    matrix = np.stack(cluster, axis=0)
    sims = np.abs(cosine_similarity(matrix, matrix))
    # exclude diagonal (self-similarity = 1)
    n = len(cluster)
    upper = sims[np.triu_indices(n, k=1)]
    return float(upper.mean())

The algorithm does not give stable components unless we use svd as initialisation.  
The components can be wrong/different from ground truth too.   

A repeating case until now is extra decomposition of a pure atom coming up as two groups.  
It might be useful to detect overlap and see if a group is a subgroup of an actual other atom in stability runs.  

I will also need to check if some atoms are "groupable". This might be a difficult problem though, i do have a matrix showing which atoms scores are correlated (we simply try to model the coefficients of atom 1 with coefficients of atom 2 * some scalar, this gives us a loss, the lower the loss, the more correlated they are).  
Among seeds, this can work similarly, we would like to see within two groups, what is the correlation.  
now its 4 atoms in group 1 vs 5 atoms in group 2, and its quite annoyiong. i can technically, do for all and use aggregates like mean/percentiles.  
This can work ig.   
Hmmmm, it might be useful to just use SVD? No, lets try to see the effect on multiple outputs i guess.  

In [ ]:
# clusters = find_stable_atoms(all_components, best_run_idx, 0.9)
cluster_idx = 6
cs = sorted(clusters[cluster_idx], key=lambda c: len(c), reverse=True)
for i, c in enumerate(cs):
    if len(c) <= 2:
        continue
    S([c.reshape(3,3) for c in cs[i]], (20,2), len(cs[i]), mode=MODE, suptitle=f"{i}: {cluster_internal_similarity(c)}", viztype="local")
    plt.show()

In [ ]:
get_live(all_runs[6][0].components).shape, all_runs[6][0].components.shape

In [ ]:
for i in range(len(all_runs[8])):
    print(i, find_dead_atoms(all_runs[8][i].components, 0.10))

In [ ]:
i = 4

S([c.reshape(3,3) for c in all_runs[8][i].components], (10,2), 8, mode=MODE)
plt.show()
lcomps = get_live(all_runs[8][i].components)
S([c.reshape(3,3) for c in lcomps], (10,2), 8, mode=MODE)
plt.show()

In [ ]:
x = sorted(hung_res.keys())
y = [hung_res[i][0] for i in x]
plt.plot(x, y)
plt.show()

In [ ]:
# comps 6 has lesser stability thqn comps 8
# lets see why
# although

In [ ]:
plot_sweep_summary(all_runs, svd_runs)

In [ ]:
runs = []
for seed in range(20):
    print("starting seed", seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    baseline_loss, loss, codes, components, recon = train_for(X, 6, 3000, seed=seed)
    all_components.append(components)

upper, stability_score, best_run_idx, pairwise_sims = hungarian_match(all_components)
clusters = find_stable_atoms(all_components, best_run_idx, 0.9)

In [ ]:
print(run.loss, run.baseline_loss)

In [ ]:
show_closest_component_of_W_for_each_component(run.components, W_true)

In [ ]:
# due to not using svd, we have some instability it seems
# svd is quite a good approximation for when the loss starts plateauing
# we will need to analyse svd initialisations behavior separately later
# it seems quite useful
plot_runs(runs)

In [ ]:
# gram error hist

# weird :) at 6,7
plt.plot([r[2] for r in runs])
plt.show()

In [ ]:
# it mostly needs more epochs
runs[6][4].shape

In [ ]:
comp7log = train_for(X, 7, epochs=3000)

In [ ]:
comp7log[4]

In [ ]:
find_dead_atoms(comp7log[4])

In [ ]:
find_dead_atoms(runs[3][4])

In [ ]:
i = 1
print(find_dead_atoms(runs[i][4]))
S([c.reshape(3,3) for c in runs[i][4]], (10,2), len(runs[i][4]), mode=MODE)

In [ ]:
# interesting, it came up with empty components
S([c.reshape(3,3) for c in comp7log[4]], (10,2), len(comp7log[4]), mode=MODE)

In [ ]:
# again weird lol
print(gram_orthogonality_error(comp7log[4].T))
show_gram(comp7log[4].T)

In [ ]:
show_gram(runs[6][4].T)

In [ ]:
# they are almost the same, at 5, even though n_components is 6
# oh 6 has index 5, all good, it has recovered exactly what it needed to
plt.plot([r[0] for r in runs])
plt.plot([r[1] for r in runs], color="red")
plt.show()

In [ ]:
for i, r in enumerate(runs):
    comps = r[3]
    S([c.reshape(3,3) for c in comps], mode=MODE, suptitle=f"{i}", ncols=len(comps), figsize=(len(comps)*5, 3))
    plt.show()

It seems this example is quite boring, the model does quite well on simple less noisy data. Lets see gram matrices

In [ ]:
# this is always satisfying in this case
for r in runs:
    show_gram(r[3].T)

# start by simple gaussian perturbation

See how loss looks.  

In [ ]:
X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    patch_dim=9, n_components=N_COMPONENTS, k=3
)
X.std(), X.max()

In [ ]:
# 10% noise, 10% std

perturb_fn = gaussian_noise_perturbation(X.std() / 10, ratio=0.1)
X, W_true, codes_true, dim_partition = perturb_fn(X, W_true, codes_true, dim_partition)

runs = []
for comp in range(1, min(N_COMPONENTS+6, 9)):
    print(f"num comps: {comp}")
    runs.append(train_for(X, comp))


plt.plot([r[0] for r in runs])
plt.plot([r[1] for r in runs], color="red")
plt.show()

In [ ]:
# recovered
show_closest_component_of_W_for_each_component(runs[5][3], W_true)

In [ ]:
# 10% samples, 33% std

perturb_fn = gaussian_noise_perturbation(X.std() / 3, ratio=0.1)
X, W_true, codes_true, dim_partition = perturb_fn(X, W_true, codes_true, dim_partition)

runs = []
for comp in range(1, min(N_COMPONENTS+6, 9)):
    print(f"num comps: {comp}")
    runs.append(train_for(X, comp))


plt.plot([r[0] for r in runs])
plt.plot([r[1] for r in runs], color="red")
plt.show()

In [ ]:
# recovered
show_closest_component_of_W_for_each_component(runs[5][3], W_true)

In [ ]:
# now we see convergence issues
# 10% samples, 300% std

perturb_fn = gaussian_noise_perturbation(3*X.std(), ratio=0.1)
X, W_true, codes_true, dim_partition = perturb_fn(X, W_true, codes_true, dim_partition)

runs = []
for comp in range(1, min(N_COMPONENTS+6, 9)):
    print(f"num comps: {comp}")
    runs.append(train_for(X, comp))


plt.plot([r[0] for r in runs])
plt.plot([r[1] for r in runs], color="red")
plt.show()

In [ ]:
# recovered
# not the best
show_closest_component_of_W_for_each_component(runs[5][3], W_true)

In [ ]:
show_closest_component_of_W_for_each_component(runs[7][3], W_true)

We see problems now, at 25% samples having 100% std, we see unstable components

In [ ]:
# 25% samples, 100% std

perturb_fn = gaussian_noise_perturbation(X.std(), ratio=0.25)
X, W_true, codes_true, dim_partition = perturb_fn(X, W_true, codes_true, dim_partition)

runs = []
for comp in range(1, min(N_COMPONENTS+6, 9)):
    print(f"num comps: {comp}")
    runs.append(train_for(X, comp))


plt.plot([r[0] for r in runs])
plt.plot([r[1] for r in runs], color="red")
plt.show()

In [ ]:
show_closest_component_of_W_for_each_component(runs[7][3], W_true)

In [ ]:
# 25% samples, 50% std
# this is abd too. the convergence actually did not happen it seems

perturb_fn = gaussian_noise_perturbation(X.std(), ratio=0.50)
X, W_true, codes_true, dim_partition = perturb_fn(X, W_true, codes_true, dim_partition)

runs = []
for comp in range(1, min(N_COMPONENTS+6, 9)):
    print(f"num comps: {comp}")
    runs.append(train_for(X, comp, epochs=3000))


plt.plot([r[0] for r in runs])
plt.plot([r[1] for r in runs], color="red")
plt.show()

In [ ]:
show_closest_component_of_W_for_each_component(runs[5][3], W_true)

In [ ]:
# 2 were not recovered
evaluate_recovery(runs[5][3], W_true, 0.9)

In [ ]:
# lets try running with different seeds for 6 components, and get the stability score
perturb_fn = gaussian_noise_perturbation(X.std(), ratio=0.25)
X, W_true, codes_true, dim_partition = perturb_fn(X, W_true, codes_true, dim_partition)

all_components = []
for seed in range(20):
    print("starting seed", seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    baseline_loss, loss, codes, components, recon = train_for(X, 6, 3000, seed=seed)
    all_components.append(components)

upper, stability_score, best_run_idx, pairwise_sims = hungarian_match(all_components)
clusters = find_stable_atoms(all_components, best_run_idx, 0.9)

In [ ]:
upper, stability_score, best_run_idx, pairwise_sims = hungarian_match(all_components)
stability_score

In [ ]:
clusters = find_stable_atoms(all_components, best_run_idx, 0.9)
cs = sorted(clusters, key=lambda c: len(c), reverse=True)

In [ ]:
S([c.reshape(3,3) for c in W_true], (8,2), len(W_true), mode=MODE)

In [ ]:
clusters = find_stable_atoms(all_components, best_run_idx, 0.9)
cs = sorted(clusters, key=lambda c: len(c), reverse=True)
for i, c in enumerate(cs):
    S([c.reshape(3,3) for c in cs[i]], (20,2), len(cs[i]), mode=MODE, suptitle=str(i))
    plt.show()